In [1]:
# ===== 0) Setup =====
from __future__ import annotations

import re
import json
import math
from statistics import mean, median, stdev
from typing import List, Dict, Any, Optional
from collections import Counter

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 200)


In [2]:
# Paths (edit if needed)
# TRAIN_FILE = "/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/train.csv"
# TEST_FILE = "/content/drive/MyDrive/git/the-ai-telco-troubleshooting-challenge/data/phase_1_test.csv"

TRAIN_FILE = "data/train.csv"
TEST_FILE = "data/phase_1_test.csv"


df_train = pd.read_csv(TRAIN_FILE)
display(df_train.head())
print(f"✓ Loaded {len(df_train)} training samples")
print(f"Columns: {df_train.columns.tolist()}")
print(f"\nFirst sample:")
print(f"Question preview: {df_train['question'].iloc[0][:]}")
print(f"Answer: {df_train['answer'].iloc[0]}")
print(f"\nAnswer distribution:")
print(df_train['answer'].value_counts())

,ID,question,answer
0,ID_1P7PJMPV0R,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,C2
1,ID_8B1D1TUTFA,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,C1
2,ID_IGGXMA9GZH,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,C2
3,ID_D6C9N2X295,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,C2
4,ID_8JC15PNP3Q,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,C5


✓ Loaded 2400 training samples
Columns: ['ID', 'question', 'answer']

First sample:
Question preview: Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and enclose its number in \boxed{{}} in the final answer.

C1: The serving cell's downtilt angle is too large, causing weak coverage at the far end.
C2: The serving cell's coverage distance exceeds 1km, resulting in over-shooting.
C3: A neighboring cell provides higher throughput.
C4: Non-colocated co-frequency neighboring cells cause severe overlapping coverage.
C5: Frequent handovers degrade performance.
C6: Neighbor cell and serving cell have the same PCI mod 30, leading to interference.
C7: Test vehicle speed exceeds 40km/h, impacting user throughput.
C8: Average scheduled RBs are below 160, affecting throughput.

Given:
- The default elect

In [3]:
# ============================================================================
# LABEL MAPPING
# ============================================================================
CAUSE_TO_NUM = {f"C{i}": i for i in range(1, 9)}
NUM_TO_CAUSE = {i: f"C{i}" for i in range(1, 9)}

# ============================================================================
# TYPE CONVERSION UTILITIES
# ============================================================================

def to_float(x) -> Optional[float]:
    """Safely convert to float, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return float(x)
    except (ValueError, TypeError):
        return None

def to_int(x) -> Optional[int]:
    """Safely convert to int, handling None, '-', and invalid values"""
    if x is None or x == "" or x == "-" or x == "—":
        return None
    try:
        return int(float(x))  # float first to handle "123.0" strings
    except (ValueError, TypeError):
        return None

def sanitize_question_text(q: str) -> str:
    """
    Convert literal '\\n' to real newlines.
    CRITICAL: Don't use unicode decoding or you corrupt '\\boxed'
    """
    if q is None:
        return ""
    return q.replace("\\n", "\n").replace("\r\n", "\n").replace("\r", "\n")

# ============================================================================
# DOMAIN-SPECIFIC CALCULATIONS
# ============================================================================

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance in meters between two lat/lon points"""
    R = 6371000.0  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def beamwidth_deg(beam_scenario: str) -> int:
    """
    Map beam scenario to vertical beamwidth:
    - DEFAULT/SCENARIO_1-5: 6 degrees
    - SCENARIO_6-11: 12 degrees
    - SCENARIO_12+: 25 degrees
    """
    s = (beam_scenario or "").upper().strip()
    if s == "DEFAULT":
        return 6

    m = re.match(r"SCENARIO_(\d+)", s)
    if not m:
        return 6

    k = int(m.group(1))
    if 1 <= k <= 5:
        return 6
    elif 6 <= k <= 11:
        return 12
    else:  # 12+
        return 25

def electronic_tilt_deg(digital_tilt) -> float:
    """
    Convert digital tilt to degrees:
    - 255 => 6 degrees (special case)
    - Otherwise: value is degrees
    """
    try:
        v = int(float(digital_tilt))
        return 6.0 if v == 255 else float(v)
    except (ValueError, TypeError):
        return 6.0  # Default fallback

print("✓ Type conversion and domain utilities loaded")

✓ Type conversion and domain utilities loaded


In [4]:
# ============================================================================
# TABLE PARSING (robust to Markdown separator rows)
# ============================================================================

def parse_pipe_table(table_text: str) -> List[Dict[str, Any]]:
    """
    Parse a pipe-delimited table into a list[dict].

    Works for common formats like:
      | Col A | Col B |
      |------:|:------|
      | 1     | 2     |

    Notes
    - Treats '-', '—', '–', '' as missing (None)
    - Ignores Markdown separator rows (e.g., '---', ':---:', '---:')
    """
    if not table_text or not str(table_text).strip():
        return []

    # keep only lines containing pipes
    raw_lines = [ln.strip() for ln in str(table_text).splitlines() if "|" in ln]
    if not raw_lines:
        return []

    def _is_separator_row(parts: List[str]) -> bool:
        # a separator row is made of dashes/colons only, e.g. '---', ':---:'
        def ok(p: str) -> bool:
            p = p.strip()
            return bool(p) and all(ch in "-: " for ch in p)
        return all(ok(p) for p in parts)

    # Normalize split: strip leading/trailing pipes then split
    def _split(line: str) -> List[str]:
        line = line.strip().strip("|")
        return [p.strip() for p in line.split("|")]

    header = _split(raw_lines[0])
    header = [h if h else f"col_{i}" for i, h in enumerate(header, start=1)]

    rows: List[Dict[str, Any]] = []
    for line in raw_lines[1:]:
        parts = _split(line)

        # drop Markdown separator lines
        if _is_separator_row(parts):
            continue

        # pad or trim to header length
        if len(parts) < len(header):
            parts += [""] * (len(header) - len(parts))
        elif len(parts) > len(header):
            parts = parts[: len(header)]

        rec = {}
        for k, v in zip(header, parts):
            v = v.strip()
            rec[k] = None if v in ("", "-", "—", "–") else v
        rows.append(rec)

    return rows

print("✓ Table parsing loaded")


✓ Table parsing loaded


In [5]:
# ============================================================================
# COLUMN MAPPING & TYPE CASTING (IMPROVED)
# ============================================================================

# Drive-test data column mappings
DRIVE_MAP = {
    "Timestamp": "timestamp",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "GPS Speed (km/h)": "gps_speed_kmh",
    "5G KPI PCell RF Serving PCI": "serving_pci",
    "5G KPI PCell RF Serving SS-RSRP [dBm]": "ss_rsrp_dbm",
    "5G KPI PCell RF Serving SS-SINR [dB]": "ss_sinr_db",
    "5G KPI PCell Layer2 MAC DL Throughput [Mbps]": "throughput_mbps",
    "5G KPI PCell Layer1 DL RB Num (Including 0)": "dl_rb_num",

    # Neighbor PCIs
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 PCI": "nei1_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 PCI": "nei2_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 PCI": "nei3_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 PCI": "nei4_pci",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 PCI": "nei5_pci",

    # Neighbor RSRPs (CRITICAL - these were missing!)
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 1 Filtered Tx BRSRP [dBm]": "nei1_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 2 Filtered Tx BRSRP [dBm]": "nei2_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 3 Filtered Tx BRSRP [dBm]": "nei3_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 4 Filtered Tx BRSRP [dBm]": "nei4_rsrp_dbm",
    "Measurement PCell Neighbor Cell Top Set(Cell Level) Top 5 Filtered Tx BRSRP [dBm]": "nei5_rsrp_dbm",
}

# Engineering parameters column mappings
ENG_MAP = {
    "gNodeB ID": "gnodeb_id",
    "Cell ID": "cell_id",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Mechanical Azimuth": "mechanical_azimuth",
    "Mechanical Downtilt": "mechanical_downtilt",
    "Digital Tilt": "digital_tilt",
    "Digital Azimuth": "digital_azimuth",
    "Beam Scenario": "beam_scenario",
    "Height": "height",
    "PCI": "pci",
    "TxRx Mode": "txrx_mode",
    "Max Transmit Power": "max_tx_power",
    "Antenna Model": "antenna_model",
}

# Define fields by type for proper casting
FLOAT_FIELDS = [
    "longitude", "latitude", "gps_speed_kmh",
    "ss_rsrp_dbm", "ss_sinr_db", "throughput_mbps", "dl_rb_num",
    "mechanical_downtilt", "digital_tilt", "height", "max_tx_power",
    "mechanical_azimuth", "digital_azimuth",
    # CRITICAL: Add neighbor RSRP fields
    "nei1_rsrp_dbm", "nei2_rsrp_dbm", "nei3_rsrp_dbm",
    "nei4_rsrp_dbm", "nei5_rsrp_dbm",
]

INT_FIELDS = [
    "pci", "serving_pci",
    "nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci",
]

def normalize_rows(rows: List[Dict], mapping: Dict[str, str]) -> List[Dict]:
    """
    Normalize column names and apply type casting.
    This ensures all numeric operations work correctly.
    """
    normalized = []

    for r in rows:
        nr = {}

        # Map column names
        for k, v in r.items():
            normalized_key = mapping.get(k)
            if normalized_key is None:
                continue  # Skip unmapped columns
            nr[normalized_key] = v

        # Type casting - CRITICAL for numeric operations
        for field in FLOAT_FIELDS:
            if field in nr:
                nr[field] = to_float(nr[field])

        for field in INT_FIELDS:
            if field in nr:
                nr[field] = to_int(nr[field])

        normalized.append(nr)

    return normalized

print("✓ Column mappings and type casting configured")
print(f"  - Float fields: {len(FLOAT_FIELDS)}")
print(f"  - Int fields: {len(INT_FIELDS)}")
print(f"  - Drive-test columns mapped: {len(DRIVE_MAP)}")
print(f"  - Engineering columns mapped: {len(ENG_MAP)}")

✓ Column mappings and type casting configured
  - Float fields: 18
  - Int fields: 7
  - Drive-test columns mapped: 19
  - Engineering columns mapped: 14


In [6]:
# ============================================================================
# RCA-FRIENDLY FEATURE ENGINEERING (STREAMLINED)
# ============================================================================


def compute_rca_features(drive_rows: List[Dict], eng_rows: List[Dict]) -> Dict[str, Any]:
    """
    Compute ONLY the features required by format_features_text.
    Optimized for efficiency - removed all unused calculations.
    
    Returns dict with 48 unique features used in the prompt.
    """
    features = {}
    
    # Helper: dBm -> mW
    def _dbm_to_mw(dbm_val: float) -> float:
        return 10 ** (dbm_val / 10.0)
    
    # Helper: safe correlation calculation
    def _safe_corr(xs, ys):
        xs2, ys2 = [], []
        for x, y in zip(xs, ys):
            if x is None or y is None:
                continue
            xs2.append(float(x))
            ys2.append(float(y))
        if len(xs2) < 2:
            return None
        mx, my = mean(xs2), mean(ys2)
        num = sum((x - mx) * (y - my) for x, y in zip(xs2, ys2))
        denx = sum((x - mx) ** 2 for x in xs2)
        deny = sum((y - my) ** 2 for y in ys2)
        if denx <= 0 or deny <= 0:
            return None
        return num / (math.sqrt(denx) * math.sqrt(deny))
    
    # -------------------------------------------------------------------------
    # C1 FEATURES: Excessive Downtilt
    # -------------------------------------------------------------------------
    serving_rsrps = [r["ss_rsrp_dbm"] for r in drive_rows if r.get("ss_rsrp_dbm") is not None]
    sinrs = [r["ss_sinr_db"] for r in drive_rows if r.get("ss_sinr_db") is not None]
    
    # rsrp_min_dbm, rsrp_mean_dbm, rsrp_10th_percentile
    if serving_rsrps:
        features["rsrp_min_dbm"] = min(serving_rsrps)
        features["rsrp_mean_dbm"] = mean(serving_rsrps)
        if len(serving_rsrps) > 2:
            features["rsrp_10th_percentile"] = sorted(serving_rsrps)[int(0.1 * len(serving_rsrps))]
        else:
            features["rsrp_10th_percentile"] = min(serving_rsrps)
    else:
        features["rsrp_min_dbm"] = None
        features["rsrp_mean_dbm"] = None
        features["rsrp_10th_percentile"] = None
    
    # sinr_degradation_db, sinr_std_when_rsrp_stable
    if serving_rsrps and sinrs and features["rsrp_mean_dbm"] is not None:
        sinr_mean = mean(sinrs)
        expected_sinr = features["rsrp_mean_dbm"] + 100
        features["sinr_degradation_db"] = expected_sinr - sinr_mean
        
        rsrp_std = stdev(serving_rsrps) if len(serving_rsrps) > 1 else 0.0
        sinr_std = stdev(sinrs) if len(sinrs) > 1 else 0.0
        denom = abs(rsrp_std) + 1e-6
        features["sinr_std_when_rsrp_stable"] = sinr_std / denom
    else:
        features["sinr_degradation_db"] = None
        features["sinr_std_when_rsrp_stable"] = None
    
    # handover_rsrp_delta_mean (computed later in handover section)
    
    # -------------------------------------------------------------------------
    # C2 FEATURES: Overshoot
    # -------------------------------------------------------------------------
    serving_pcis = [r["serving_pci"] for r in drive_rows if r.get("serving_pci") is not None]
    pci_to_cell = {c["pci"]: c for c in eng_rows if c.get("pci") is not None}
    
    # dist_max_m, dist_p95_m, overshoot_flag
    distances = []
    for r in drive_rows:
        spci = r.get("serving_pci")
        cell = pci_to_cell.get(spci)
        
        if not cell:
            continue
        
        if (r.get("latitude") is None or r.get("longitude") is None or
            cell.get("latitude") is None or cell.get("longitude") is None):
            continue
        
        dist = haversine_m(
            r["latitude"], r["longitude"],
            cell["latitude"], cell["longitude"]
        )
        distances.append(dist)
    
    if distances:
        features["dist_max_m"] = max(distances)
        features["dist_p95_m"] = sorted(distances)[int(0.95 * (len(distances) - 1))]
        features["overshoot_flag"] = features["dist_p95_m"] > 1000 if features["dist_p95_m"] else False
    else:
        features["dist_max_m"] = None
        features["dist_p95_m"] = None
        features["overshoot_flag"] = False
    
    # handover_count
    handover_count = 0
    for i in range(1, len(serving_pcis)):
        if serving_pcis[i] != serving_pcis[i - 1]:
            handover_count += 1
    features["handover_count"] = handover_count
    
    # throughput_variance_across_cells, best_cell_throughput_gap (computed later in C3)
    
    # -------------------------------------------------------------------------
    # C3 FEATURES: Wrong Cell Selection
    # -------------------------------------------------------------------------
    tps = [r["throughput_mbps"] for r in drive_rows if r.get("throughput_mbps") is not None]
    
    # tp_samples_below_600, tp_drop_ratio
    if tps:
        features["tp_samples_below_600"] = sum(1 for x in tps if x < 600)
        features["tp_drop_ratio"] = sum(1 for x in tps if x < 600) / len(tps)
    else:
        features["tp_samples_below_600"] = 0
        features["tp_drop_ratio"] = None
    
    # Handover TP deltas and cross-cell throughput comparison
    ho_tp_deltas = []
    ho_rsrp_deltas = []
    cell_throughputs = {}
    
    prev = None
    for r in drive_rows:
        # Track throughput by PCI
        pci = r.get("serving_pci")
        tp = r.get("throughput_mbps")
        if pci and tp:
            cell_throughputs.setdefault(pci, []).append(tp)
        
        # Handover deltas
        if prev is None:
            prev = r
            continue
        
        prev_pci = prev.get("serving_pci")
        curr_pci = r.get("serving_pci")
        
        if prev_pci is not None and curr_pci is not None and curr_pci != prev_pci:
            prev_tp = prev.get("throughput_mbps")
            curr_tp = r.get("throughput_mbps")
            prev_rsrp = prev.get("ss_rsrp_dbm")
            curr_rsrp = r.get("ss_rsrp_dbm")
            
            if prev_tp is not None and curr_tp is not None:
                ho_tp_deltas.append(curr_tp - prev_tp)
            if prev_rsrp is not None and curr_rsrp is not None:
                ho_rsrp_deltas.append(curr_rsrp - prev_rsrp)
        
        prev = r
    
    features["handover_tp_delta_mean"] = mean(ho_tp_deltas) if ho_tp_deltas else None
    features["handover_rsrp_delta_mean"] = mean(ho_rsrp_deltas) if ho_rsrp_deltas else None
    features["handover_improves_tp_flag"] = (features["handover_tp_delta_mean"] is not None and 
                                             features["handover_tp_delta_mean"] > 80)
    
    # avg_throughput_change_after_handover, throughput_improved_by_handover
    features["avg_throughput_change_after_handover"] = mean(ho_tp_deltas) if ho_tp_deltas else 0.0
    features["throughput_improved_by_handover"] = features["avg_throughput_change_after_handover"] > 50
    
    # throughput_variance_across_cells, best_cell_throughput_gap
    if len(cell_throughputs) >= 2:
        cell_avg_tp = {pci: mean(tps) for pci, tps in cell_throughputs.items()}
        
        if serving_pcis:
            most_common_pci = Counter(serving_pcis).most_common(1)[0][0]
            current_avg = cell_avg_tp.get(most_common_pci, 0)
            other_avgs = [avg for pci, avg in cell_avg_tp.items() if pci != most_common_pci]
            best_alt = max(other_avgs) if other_avgs else 0
            features["best_cell_throughput_gap"] = best_alt - current_avg
        else:
            features["best_cell_throughput_gap"] = 0.0
        
        best_tp_pci = max(cell_avg_tp.items(), key=lambda x: x[1])
        worst_tp_pci = min(cell_avg_tp.items(), key=lambda x: x[1])
        features["throughput_variance_across_cells"] = best_tp_pci[1] - worst_tp_pci[1]
    else:
        features["best_cell_throughput_gap"] = 0.0
        features["throughput_variance_across_cells"] = None
    
    # -------------------------------------------------------------------------
    # C4 FEATURES: Overlapping Coverage
    # -------------------------------------------------------------------------
    pci_to_gnodeb = {}
    for cell in eng_rows:
        if cell.get("pci") is not None:
            pci_to_gnodeb[cell["pci"]] = cell.get("gnodeb_id")
    
    # Get serving cell's gNodeB
    serving_gnodeb = None
    if serving_pcis:
        serving_pci = serving_pcis[-1]
        serving_gnodeb = pci_to_gnodeb.get(serving_pci)
    
    # noncolocated_strong_neighbor_gnodeb_count_mean, noncolocated_strong_neighbor_gnodeb_count_max
    noncolocated_strong_counts = []
    coloc_strong = 0
    noncoloc_strong = 0
    top1_top2_gaps = []
    
    neighbor_rsrp_by_pci = {}
    
    for r in drive_rows:
        serv_pci = r.get("serving_pci")
        serv_rsrp = r.get("ss_rsrp_dbm")
        if serv_pci is None or serv_rsrp is None:
            continue
        
        serv_gnb = pci_to_gnodeb.get(serv_pci)
        if serv_gnb is None:
            continue
        
        # Track strong non-colocated neighbors per sample
        strong_noncolocated_gnbs = set()
        nei_vals = []
        
        for i in range(1, 6):
            nei_pci = r.get(f"nei{i}_pci")
            nei_rsrp = r.get(f"nei{i}_rsrp_dbm")
            
            if nei_pci is None or nei_rsrp is None:
                continue
            
            # Track for neighbor analysis
            if nei_pci not in neighbor_rsrp_by_pci:
                neighbor_rsrp_by_pci[nei_pci] = []
            neighbor_rsrp_by_pci[nei_pci].append(nei_rsrp)
            nei_vals.append(nei_rsrp)
            
            # Strong neighbor: RSRP > -95 AND within 6 dB of serving
            if nei_rsrp > -95 and abs(nei_rsrp - serv_rsrp) <= 6:
                nei_gnb = pci_to_gnodeb.get(nei_pci)
                if nei_gnb is not None:
                    if nei_gnb != serv_gnb:
                        strong_noncolocated_gnbs.add(nei_gnb)
                        noncoloc_strong += 1
                    else:
                        coloc_strong += 1
        
        noncolocated_strong_counts.append(len(strong_noncolocated_gnbs))
        
        # top1_top2_gap_db_mean
        if len(nei_vals) >= 2:
            sorted_nei = sorted(nei_vals, reverse=True)
            top1, top2 = sorted_nei[0], sorted_nei[1]
            top1_top2_gaps.append(top1 - top2)
    
    features["noncolocated_strong_neighbor_gnodeb_count_mean"] = (mean(noncolocated_strong_counts) 
                                                                    if noncolocated_strong_counts else 0.0)
    features["noncolocated_strong_neighbor_gnodeb_count_max"] = (max(noncolocated_strong_counts) 
                                                                  if noncolocated_strong_counts else 0)
    features["top1_top2_gap_db_mean"] = mean(top1_top2_gaps) if top1_top2_gaps else None
    
    # strong_neighbor_noncolocated_share
    denom = coloc_strong + noncoloc_strong
    features["strong_neighbor_noncolocated_share"] = (noncoloc_strong / denom) if denom > 0 else 0.0
    
    # high_interference_power_ratio_flag
    ratios_db = []
    for r in drive_rows:
        serv_rsrp = r.get("ss_rsrp_dbm")
        if serv_rsrp is None:
            continue
        S = _dbm_to_mw(serv_rsrp)
        I = 0.0
        for i in range(1, 6):
            nei_rsrp = r.get(f"nei{i}_rsrp_dbm")
            if nei_rsrp is None:
                continue
            if nei_rsrp > -95 and abs(nei_rsrp - serv_rsrp) <= 6:
                I += _dbm_to_mw(nei_rsrp)
        if S > 0 and I > 0:
            ratios_db.append(10.0 * math.log10((I + 1e-12) / (S + 1e-12)))
    
    if ratios_db:
        ratios_db_sorted = sorted(ratios_db)
        p90_idx = int(0.9 * (len(ratios_db_sorted) - 1))
        features["high_interference_power_ratio_flag"] = ratios_db_sorted[p90_idx] > -3.0
    else:
        features["high_interference_power_ratio_flag"] = False
    
    # rsrp_advantage_of_best_neighbor
    best_neighbor_rsrp = None
    if neighbor_rsrp_by_pci:
        best_pci = max(neighbor_rsrp_by_pci.items(), key=lambda x: mean(x[1]))
        best_neighbor_rsrp = mean(best_pci[1])
    
    serving_rsrp_avg = mean(serving_rsrps) if serving_rsrps else None
    if best_neighbor_rsrp is not None and serving_rsrp_avg is not None:
        features["rsrp_advantage_of_best_neighbor"] = best_neighbor_rsrp - serving_rsrp_avg
    else:
        features["rsrp_advantage_of_best_neighbor"] = None
    
    # -------------------------------------------------------------------------
    # C5 FEATURES: Ping-Pong Handover
    # -------------------------------------------------------------------------
    # ping_pong_handover_count, ping_pong_detected, frequent_handover_flag
    ping_pong_events = 0
    prev_pci = None
    prev_prev_pci = None
    
    for r in drive_rows:
        curr_pci = r.get("serving_pci")
        if prev_pci and prev_prev_pci and curr_pci == prev_prev_pci and curr_pci != prev_pci:
            ping_pong_events += 1
        prev_prev_pci = prev_pci
        prev_pci = curr_pci
    
    features["ping_pong_handover_count"] = ping_pong_events
    features["ping_pong_detected"] = ping_pong_events >= 2
    features["frequent_handover_flag"] = handover_count >= 3
    
    # Note: handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean already computed above
    
    # -------------------------------------------------------------------------
    # C6 FEATURES: PCI Collision
    # -------------------------------------------------------------------------
    # serving_electronic_tilt_deg, serving_total_tilt_deg, tilt_to_beamwidth_ratio
    # noncolocated_neighbor_count, colocated_neighbor_count, abnormal_path_loss
    
    if serving_pcis:
        cell = pci_to_cell.get(serving_pcis[-1])
        
        if cell:
            mech_tilt = cell.get("mechanical_downtilt") or 0.0
            elec_tilt = electronic_tilt_deg(cell.get("digital_tilt"))
            total_tilt = float(mech_tilt) + float(elec_tilt)
            
            vbw = beamwidth_deg(cell.get("beam_scenario", "DEFAULT"))
            tilt_to_beamwidth_ratio = total_tilt / vbw if vbw > 0 else 0.0
            
            features["serving_electronic_tilt_deg"] = elec_tilt
            features["serving_total_tilt_deg"] = total_tilt
            features["tilt_to_beamwidth_ratio"] = tilt_to_beamwidth_ratio
        else:
            features["serving_electronic_tilt_deg"] = None
            features["serving_total_tilt_deg"] = None
            features["tilt_to_beamwidth_ratio"] = None
    else:
        features["serving_electronic_tilt_deg"] = None
        features["serving_total_tilt_deg"] = None
        features["tilt_to_beamwidth_ratio"] = None
    
    # noncolocated_neighbor_count, colocated_neighbor_count
    neighbor_gnodebs = set()
    colocated_neighbors = []
    noncolocated_neighbors = []
    
    for r in drive_rows:
        for k in ["nei1_pci", "nei2_pci", "nei3_pci", "nei4_pci", "nei5_pci"]:
            nei_pci = r.get(k)
            if nei_pci is not None:
                nei_gnodeb = pci_to_gnodeb.get(nei_pci)
                if nei_gnodeb is not None:
                    neighbor_gnodebs.add(nei_gnodeb)
                    if serving_gnodeb is not None:
                        if nei_gnodeb == serving_gnodeb:
                            colocated_neighbors.append(nei_pci)
                        else:
                            noncolocated_neighbors.append(nei_pci)
    
    features["noncolocated_neighbor_count"] = len(set(noncolocated_neighbors))
    features["colocated_neighbor_count"] = len(set(colocated_neighbors))
    
    # abnormal_path_loss
    if distances and serving_rsrps and len(distances) == len(serving_rsrps):
        path_loss_deviations = []
        for i, (dist, rsrp) in enumerate(zip(distances, serving_rsrps)):
            if dist > 10:
                dist_km = dist / 1000
                if dist_km > 0:
                    expected_pl = 128.1 + 37.6 * math.log10(dist_km)
                    actual_pl = 46 - rsrp
                    deviation = abs(actual_pl - expected_pl)
                    path_loss_deviations.append(deviation)
        
        if path_loss_deviations:
            path_loss_deviation_mean = mean(path_loss_deviations)
            features["abnormal_path_loss"] = path_loss_deviation_mean > 15
        else:
            features["abnormal_path_loss"] = False
    else:
        features["abnormal_path_loss"] = False
    
    # -------------------------------------------------------------------------
    # C7 FEATURES: High Speed
    # -------------------------------------------------------------------------
    speeds = [r["gps_speed_kmh"] for r in drive_rows if r.get("gps_speed_kmh") is not None]
    
    if speeds:
        features["speed_max_kmh"] = max(speeds)
        features["speed_mean_kmh"] = mean(speeds)
        features["speed_above_40_flag"] = features["speed_max_kmh"] > 40 if features["speed_max_kmh"] else False
    else:
        features["speed_max_kmh"] = None
        features["speed_mean_kmh"] = None
        features["speed_above_40_flag"] = False
    
    # fast_low_tp_ratio, speed_tp_correlation, c7_speed_hint
    speed_tp_pairs = []
    fast_total = 0
    fast_low_tp = 0
    
    for r in drive_rows:
        spd = r.get("gps_speed_kmh")
        tp = r.get("throughput_mbps")
        if spd is None or tp is None:
            continue
        speed_tp_pairs.append((spd, tp))
        if spd >= 40:
            fast_total += 1
            if tp < 600:
                fast_low_tp += 1
    
    features["fast_low_tp_ratio"] = (fast_low_tp / fast_total) if fast_total > 0 else 0.0
    
    if speed_tp_pairs:
        speeds2 = [x for x, _ in speed_tp_pairs]
        tps2 = [y for _, y in speed_tp_pairs]
        features["speed_tp_correlation"] = _safe_corr(speeds2, tps2)
    else:
        features["speed_tp_correlation"] = None
    
    features["c7_speed_hint"] = (
        features.get("speed_above_40_flag", False) and
        features.get("fast_low_tp_ratio", 0.0) > 0.25
    )
    
    # -------------------------------------------------------------------------
    # C8 FEATURES: Resource Congestion
    # -------------------------------------------------------------------------
    rbs = [r["dl_rb_num"] for r in drive_rows if r.get("dl_rb_num") is not None]
    
    if rbs:
        features["rb_mean"] = mean(rbs)
        features["rb_min"] = min(rbs)
    else:
        features["rb_mean"] = None
        features["rb_min"] = None
    
    # high_rb_low_tp_ratio_v2, tp_drop_with_high_rb_ratio, rb_tp_correlation
    rb_tp_pairs2 = []
    high_rb_total = 0
    high_rb_low_tp = 0
    drop_total = 0
    drop_high_rb = 0
    
    for r in drive_rows:
        rb = r.get("dl_rb_num")
        tp = r.get("throughput_mbps")
        if rb is None or tp is None:
            continue
        
        rb_tp_pairs2.append((rb, tp))
        
        if rb >= 180:
            high_rb_total += 1
            if tp < 600:
                high_rb_low_tp += 1
        
        if tp < 600:
            drop_total += 1
            if rb > 180:
                drop_high_rb += 1
    
    features["high_rb_low_tp_ratio_v2"] = (high_rb_low_tp / high_rb_total) if high_rb_total > 0 else 0.0
    features["tp_drop_with_high_rb_ratio"] = (drop_high_rb / drop_total) if drop_total > 0 else 0.0
    
    if rb_tp_pairs2:
        rbs2 = [x for x, _ in rb_tp_pairs2]
        tps3 = [y for _, y in rb_tp_pairs2]
        features["rb_tp_correlation"] = _safe_corr(rbs2, tps3)
    else:
        features["rb_tp_correlation"] = None
    
    # throughput_efficiency_min
    tp_rb_pairs = [(r.get("throughput_mbps"), r.get("dl_rb_num"))
                   for r in drive_rows
                   if r.get("throughput_mbps") is not None and r.get("dl_rb_num") is not None and r.get("dl_rb_num") > 0]
    
    if tp_rb_pairs:
        efficiencies = [tp / rb for tp, rb in tp_rb_pairs]
        features["throughput_efficiency_min"] = min(efficiencies)
    else:
        features["throughput_efficiency_min"] = None
    
    return features

print("✓ RCA feature engineering ready (STREAMLINED)")
print("  Computes ONLY 48 features required by format_features_text:")
print("    C1 (6): rsrp_min_dbm, rsrp_10th_percentile, handover_rsrp_delta_mean, rsrp_mean_dbm,")
print("            sinr_degradation_db, sinr_std_when_rsrp_stable")
print("    C2 (6): dist_max_m, overshoot_flag, dist_p95_m, throughput_variance_across_cells,")
print("            handover_count, best_cell_throughput_gap")
print("    C3 (6): avg_throughput_change_after_handover, handover_tp_delta_mean, handover_improves_tp_flag,")
print("            throughput_improved_by_handover, tp_samples_below_600, tp_drop_ratio")
print("    C4 (6): noncolocated_strong_neighbor_gnodeb_count_mean, strong_neighbor_noncolocated_share,")
print("            noncolocated_strong_neighbor_gnodeb_count_max, high_interference_power_ratio_flag,")
print("            rsrp_advantage_of_best_neighbor, top1_top2_gap_db_mean")
print("    C5 (6): ping_pong_handover_count, frequent_handover_flag, ping_pong_detected,")
print("            handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean")
print("    C6 (6): serving_electronic_tilt_deg, serving_total_tilt_deg, noncolocated_neighbor_count,")
print("            abnormal_path_loss, tilt_to_beamwidth_ratio, colocated_neighbor_count")
print("    C7 (6): speed_max_kmh, c7_speed_hint, speed_above_40_flag, fast_low_tp_ratio,")
print("            speed_mean_kmh, speed_tp_correlation")
print("    C8 (6): rb_mean, high_rb_low_tp_ratio_v2, rb_min, tp_drop_with_high_rb_ratio,")
print("            rb_tp_correlation, throughput_efficiency_min")


✓ RCA feature engineering ready (STREAMLINED)
  Computes ONLY 48 features required by format_features_text:
    C1 (6): rsrp_min_dbm, rsrp_10th_percentile, handover_rsrp_delta_mean, rsrp_mean_dbm,
            sinr_degradation_db, sinr_std_when_rsrp_stable
    C2 (6): dist_max_m, overshoot_flag, dist_p95_m, throughput_variance_across_cells,
            handover_count, best_cell_throughput_gap
    C3 (6): avg_throughput_change_after_handover, handover_tp_delta_mean, handover_improves_tp_flag,
            throughput_improved_by_handover, tp_samples_below_600, tp_drop_ratio
    C4 (6): noncolocated_strong_neighbor_gnodeb_count_mean, strong_neighbor_noncolocated_share,
            noncolocated_strong_neighbor_gnodeb_count_max, high_interference_power_ratio_flag,
            rsrp_advantage_of_best_neighbor, top1_top2_gap_db_mean
    C5 (6): ping_pong_handover_count, frequent_handover_flag, ping_pong_detected,
            handover_count, handover_rsrp_delta_mean, handover_tp_delta_mean
    C6

In [7]:
# ============================================================================
# COMPLETE FEATURE EXTRACTION PIPELINE
# ============================================================================

def extract_all_features(question_text: str) -> Dict[str, Any]:
    """
    Complete pipeline to extract features from raw question text.
    
    Steps:
    1. Parse drive test table and engineering table
    2. Map column names to standardized format
    3. Convert types appropriately
    4. Compute RCA features
    
    Returns: Dictionary with all 48 RCA features
    """
    # Split question into drive test and engineering sections
    parts = question_text.split("\n\n")
    drive_text = ""
    eng_text = ""
    
    for part in parts:
        if "Timestamp" in part and "Longitude" in part:
            drive_text = part
        elif "PCI" in part and "gNodeB_ID" in part:
            eng_text = part
    
    # Parse tables
    drive_rows_raw = parse_pipe_table(drive_text)
    eng_rows_raw = parse_pipe_table(eng_text)
    
    # Map column names and convert types
    drive_rows = []
    for row_raw in drive_rows_raw:
        row = {}
        for old_col, new_col in DRIVE_MAP.items():
            val = row_raw.get(old_col)
            if new_col in FLOAT_FIELDS:
                row[new_col] = to_float(val)
            elif new_col in INT_FIELDS:
                row[new_col] = to_int(val)
            else:
                row[new_col] = val
        drive_rows.append(row)
    
    eng_rows = []
    for row_raw in eng_rows_raw:
        row = {}
        for old_col, new_col in ENG_MAP.items():
            val = row_raw.get(old_col)
            if new_col in FLOAT_FIELDS:
                row[new_col] = to_float(val)
            elif new_col in INT_FIELDS:
                row[new_col] = to_int(val)
            else:
                row[new_col] = val
        eng_rows.append(row)
    
    # Compute features
    features_dict = compute_rca_features(drive_rows, eng_rows)
    
    # Also store PCI for reference
    if drive_rows:
        serving_pcis = [r.get("serving_pci") for r in drive_rows if r.get("serving_pci") is not None]
        if serving_pcis:
            features_dict["serving_pci"] = serving_pcis[-1]  # Last serving PCI
    
    return features_dict

print("✓ Complete feature extraction pipeline ready")
print("  Usage: features = extract_all_features(question_text)")
print("  Returns: Dict with 48 RCA features + serving_pci")

✓ Complete feature extraction pipeline ready
  Usage: features = extract_all_features(question_text)
  Returns: Dict with 48 RCA features + serving_pci


In [8]:
# ============================================================================
# ENHANCED QUESTION FORMATTING (Q&A Format)
# ============================================================================

def format_value(v) -> str:
    """Format value for display in prompt"""
    if v is None:
        return "N/A"
    if isinstance(v, bool):
        return "Yes" if v else "No"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return f"{v:.2f}"
    return str(v)

def format_features_text(features: Dict) -> str:
    """
    Compact format focusing on top discriminating features per class.
    Optimized for token efficiency while preserving classification power.
    """
    feature_text = "\n".join([
        "**Key RCA Features:**",
        "",
        "**C1 Indicators (Excessive Downtilt):**",
        "  1. RSRP min: {0} dBm".format(format_value(features.get('rsrp_min_dbm'))),
        "  2. RSRP 10th percentile: {0} dBm".format(format_value(features.get('rsrp_10th_percentile'))),
        "  3. Handover RSRP delta mean: {0} dB".format(format_value(features.get('handover_rsrp_delta_mean'))),
        "  4. RSRP mean: {0} dBm".format(format_value(features.get('rsrp_mean_dbm'))),
        "  5. SINR degradation: {0} dB".format(format_value(features.get('sinr_degradation_db'))),
        "  6. SINR std when RSRP stable: {0}".format(format_value(features.get('sinr_std_when_rsrp_stable'))),
        "",
        "**C2 Indicators (Overshoot):**",
        "  1. Distance max: {0} m".format(format_value(features.get('dist_max_m'))),
        "  2. Overshoot flag: {0}".format(format_value(features.get('overshoot_flag'))),
        "  3. Distance p95: {0} m".format(format_value(features.get('dist_p95_m'))),
        "  4. TP variance across cells: {0}".format(format_value(features.get('throughput_variance_across_cells'))),
        "  5. Handover count: {0}".format(format_value(features.get('handover_count'))),
        "  6. Best cell TP gap: {0} Mbps".format(format_value(features.get('best_cell_throughput_gap'))),
        "",
        "**C3 Indicators (Wrong Cell Selection):**",
        "  1. Avg TP change after handover: {0} Mbps".format(format_value(features.get('avg_throughput_change_after_handover'))),
        "  2. Handover TP delta mean: {0} Mbps".format(format_value(features.get('handover_tp_delta_mean'))),
        "  3. Handover improves TP flag: {0}".format(format_value(features.get('handover_improves_tp_flag'))),
        "  4. TP improved by handover: {0}".format(format_value(features.get('throughput_improved_by_handover'))),
        "  5. TP samples below 600: {0}".format(format_value(features.get('tp_samples_below_600'))),
        "  6. TP drop ratio: {0}".format(format_value(features.get('tp_drop_ratio'))),
        "",
        "**C4 Indicators (Overlapping Coverage):**",
        "  1. Non-colocated strong neighbor gNodeB count mean: {0}".format(format_value(features.get('noncolocated_strong_neighbor_gnodeb_count_mean'))),
        "  2. Strong neighbor non-colocated share: {0}".format(format_value(features.get('strong_neighbor_noncolocated_share'))),
        "  3. Non-colocated strong neighbor gNodeB count max: {0}".format(format_value(features.get('noncolocated_strong_neighbor_gnodeb_count_max'))),
        "  4. High interference power ratio flag: {0}".format(format_value(features.get('high_interference_power_ratio_flag'))),
        "  5. RSRP advantage of best neighbor: {0} dB".format(format_value(features.get('rsrp_advantage_of_best_neighbor'))),
        "  6. Top1-top2 gap dB mean: {0} dB".format(format_value(features.get('top1_top2_gap_db_mean'))),
        "",
        "**C5 Indicators (Ping-Pong Handover):**",
        "  1. Ping-pong handover count: {0}".format(format_value(features.get('ping_pong_handover_count'))),
        "  2. Frequent handover flag: {0}".format(format_value(features.get('frequent_handover_flag'))),
        "  3. Ping-pong detected: {0}".format(format_value(features.get('ping_pong_detected'))),
        "  4. Handover count: {0}".format(format_value(features.get('handover_count'))),
        "  5. Handover RSRP delta mean: {0} dB".format(format_value(features.get('handover_rsrp_delta_mean'))),
        "  6. Handover TP delta mean: {0} Mbps".format(format_value(features.get('handover_tp_delta_mean'))),
        "",
        "**C6 Indicators (PCI Collision):**",
        "  1. Serving electronic tilt: {0}°".format(format_value(features.get('serving_electronic_tilt_deg'))),
        "  2. Serving total tilt: {0}°".format(format_value(features.get('serving_total_tilt_deg'))),
        "  3. Non-colocated neighbor count: {0}".format(format_value(features.get('noncolocated_neighbor_count'))),
        "  4. Abnormal path loss: {0}".format(format_value(features.get('abnormal_path_loss'))),
        "  5. Tilt to beamwidth ratio: {0}".format(format_value(features.get('tilt_to_beamwidth_ratio'))),
        "  6. Co-located neighbor count: {0}".format(format_value(features.get('colocated_neighbor_count'))),
        "",
        "**C7 Indicators (High Speed):**",
        "  1. Speed max: {0} km/h".format(format_value(features.get('speed_max_kmh'))),
        "  2. C7 speed hint: {0}".format(format_value(features.get('c7_speed_hint'))),
        "  3. Speed above 40 flag: {0}".format(format_value(features.get('speed_above_40_flag'))),
        "  4. Fast low TP ratio: {0}".format(format_value(features.get('fast_low_tp_ratio'))),
        "  5. Speed mean: {0} km/h".format(format_value(features.get('speed_mean_kmh'))),
        "  6. Speed-TP correlation: {0}".format(format_value(features.get('speed_tp_correlation'))),
        "",
        "**C8 Indicators (Resource Congestion):**",
        "  1. RB mean: {0}".format(format_value(features.get('rb_mean'))),
        "  2. High RB low TP ratio v2: {0}".format(format_value(features.get('high_rb_low_tp_ratio_v2'))),
        "  3. RB min: {0}".format(format_value(features.get('rb_min'))),
        "  4. TP drop with high RB ratio: {0}".format(format_value(features.get('tp_drop_with_high_rb_ratio'))),
        "  5. RB-TP correlation: {0}".format(format_value(features.get('rb_tp_correlation'))),
        "  6. TP efficiency min: {0}".format(format_value(features.get('throughput_efficiency_min'))),
    ])

    return feature_text

def format_enhanced_question(original_question: str, features_text: str) -> str:

    """

    Combine the original question with engineered features.print("  - format_enhanced_question(): Combines original question + features")

    This creates the enhanced question column WITHOUT the answer.print("  - format_features_text(): Formats features into readable text")

    """
    print("✓ Q&A format functions ready")

    enhanced_question = f"""
        {original_question}
        {features_text}
        """

    return enhanced_question

In [9]:
def safe_format(val, unit="", decimals=1):
    """Safely format feature values, handling None/missing data"""
    if val is None:
        return "N/A"
    try:
        if isinstance(val, (int, float)):
            if decimals == 0:
                return f"{int(val)}{unit}"
            return f"{float(val):.{decimals}f}{unit}"
        return str(val)
    except (ValueError, TypeError):
        return "N/A"

def get_relevant_features(answer: str, features: Dict) -> Dict:
    """Extract only features relevant to specific root cause class"""
    # Common features for all classes
    common = {
        'rsrp_mean_dbm': features.get('rsrp_mean_dbm'),
        'sinr_mean_db': features.get('sinr_mean_db')
    }
    
    # Class-specific feature subsets
    class_features = {
        "C1": ['rsrp_min_dbm', 'rsrp_10th_percentile', 'sinr_degradation_db', 
               'serving_total_tilt_deg', 'dist_p95_m', 'rsrp_advantage_of_best_neighbor'],
        "C2": ['dist_max_m', 'dist_p95_m', 'overshoot_flag', 'handover_count',
               'throughput_variance_across_cells', 'best_cell_throughput_gap'],
        "C3": ['avg_throughput_change_after_handover', 'handover_tp_delta_mean',
               'handover_improves_tp_flag', 'tp_samples_below_600', 'tp_drop_ratio'],
        "C4": ['noncolocated_strong_neighbor_gnodeb_count_mean', 'strong_neighbor_noncolocated_share',
               'top1_top2_gap_db_mean', 'high_interference_power_ratio_flag', 'rsrp_advantage_of_best_neighbor'],
        "C5": ['ping_pong_handover_count', 'frequent_handover_flag', 'ping_pong_detected',
               'handover_count', 'handover_rsrp_delta_mean', 'handover_tp_delta_mean'],
        "C6": ['noncolocated_neighbor_count', 'colocated_neighbor_count', 'serving_pci',
               'abnormal_path_loss', 'serving_electronic_tilt_deg'],
        "C7": ['speed_max_kmh', 'c7_speed_hint', 'speed_above_40_flag',
               'fast_low_tp_ratio', 'speed_mean_kmh', 'speed_tp_correlation'],
        "C8": ['rb_mean', 'high_rb_low_tp_ratio_v2', 'tp_drop_with_high_rb_ratio',
               'rb_tp_correlation', 'throughput_efficiency_min', 'rb_min']
    }
    
    relevant = common.copy()
    for feat in class_features.get(answer, []):
        if feat in features:
            relevant[feat] = features[feat]
    
    return relevant

def format_optimal_training_prompt(row: Dict) -> Dict:
    """
    Creates optimal training format for Qwen 1.5B with:
    - COMPACT system prompt (~100 tokens)
    - BRIEF reasoning (max 150 tokens)
    - FILTERED features (only relevant per class)
    - Clear decision rules
    """

    # Extract the engineered features
    features = row['features_dict']
    original_q = row['original_question']
    answer = row['answer']

    # Build COMPACT system prompt optimized for 1.5B model (~100 tokens)
    EXPERT_SYSTEM = """You are a 5G RAN specialist. Analyze systematically:
1. RSRP/SINR patterns 2. Neighbor relations 3. Handover behavior 4. Speed/resources

Classes: C1=Downtilt (weak edge), C2=Overshoot (>1km), C3=WrongCell (HO improves TP), C4=Interference (good RSRP+poor SINR), C5=PingPong (frequent HO), C6=PCI collision, C7=Speed (>40km/h), C8=Congestion (high RB+low TP)

Output: Brief reasoning + \\boxed{{n}}"""

    # Build reasoning-enhanced user prompt
    USER_PROMPT = f"""{original_q}

{row['features_text']}

**Analysis Instructions:**
Think step-by-step:

1. **Signal Quality Assessment:**
   - What are the RSRP and SINR levels?
   - Are they correlated (both low = C1/C3) or decoupled (good RSRP + poor SINR = C4)?

2. **Neighbor Analysis:**
   - How many strong neighbors exist?
   - Are they from the same gNodeB (co-located) or different sites (non-co-located)?
   - Is there one dominant neighbor (C3) or multiple equal neighbors (C4)?

3. **Mobility & Handover:**
   - How many handovers occurred?
   - Do handovers improve throughput (C3/C5) or not?
   - Is there ping-pong behavior (back-and-forth)?

4. **Efficiency Metrics:**
   - Is RB allocation high but throughput low (C8)?
   - Does speed correlate with poor performance (C7)?
   - Is there abnormal path loss or tilt issues (C1)?

5. **Distance & Coverage:**
   - Is the UE very far from the cell (C2)?
   - Are there coverage holes or weak spots (C1)?

Based on this systematic analysis, identify the PRIMARY root cause."""

    # Build reasoning-based assistant response with COMPACT formatting
    ASSISTANT_REASONING = generate_reasoning_for_class(answer, features)

    ASSISTANT_RESPONSE = f"""{ASSISTANT_REASONING}

**Final Answer:** \\boxed{{{answer[1]}}}"""  # Extract number from "C1" -> "1"

    return {
        "messages": [
            {"role": "system", "content": EXPERT_SYSTEM},
            {"role": "user", "content": USER_PROMPT},
            {"role": "assistant", "content": ASSISTANT_RESPONSE}
        ]
    }


def generate_reasoning_for_class(answer: str, features: Dict) -> str:
    """
    Generate COMPACT class-specific reasoning (max 150 tokens).
    Optimized for 1.5B model - filters only relevant features.
    """
    # Filter to only relevant features for this class
    relevant = get_relevant_features(answer, features)
    
    # Extract key metrics with safe formatting
    rsrp_mean = safe_format(relevant.get('rsrp_mean_dbm'), ' dBm')
    sinr_mean = safe_format(relevant.get('sinr_mean_db'), ' dB')

    # COMPACT reasoning templates (max 150 tokens each)
    reasoning_templates = {
        "C1": f"""**Analysis:**
• Signal: RSRP={rsrp_mean} (weak edge), SINR={sinr_mean}
• Coverage: RSRP min={safe_format(relevant.get('rsrp_min_dbm'), ' dBm')}, p10={safe_format(relevant.get('rsrp_10th_percentile'), ' dBm')} → poor far-end
• Geometry: Tilt={safe_format(relevant.get('serving_total_tilt_deg'), '°')}, dist_p95={safe_format(relevant.get('dist_p95_m'), 'm')}
• SINR degrades {safe_format(relevant.get('sinr_degradation_db'), ' dB')} when RSRP stable
• No dominant better neighbor (advantage: {safe_format(relevant.get('rsrp_advantage_of_best_neighbor'), ' dB')})
**Root Cause:** C1 - Excessive downtilt causing weak far coverage""",

        "C2": f"""**Analysis:**
• Distance: max={safe_format(relevant.get('dist_max_m'), 'm')}, p95={safe_format(relevant.get('dist_p95_m'), 'm')} → overshoot (>1000m)
• Overshoot flag: {safe_format(relevant.get('overshoot_flag'))}
• Handovers: {safe_format(relevant.get('handover_count'), '', 0)} times
• TP variance across cells: {safe_format(relevant.get('throughput_variance_across_cells'), ' Mbps')} → multiple cells visible
• Best cell TP gap: {safe_format(relevant.get('best_cell_throughput_gap'), ' Mbps')}
**Root Cause:** C2 - Overshoot (serving cell too far, coverage boundary issue)""",

        "C3": f"""**Analysis:**
• Handover impact: TP change after HO = {safe_format(relevant.get('avg_throughput_change_after_handover'), ' Mbps')} (significant improvement)
• HO TP delta: {safe_format(relevant.get('handover_tp_delta_mean'), ' Mbps')} → improves throughput: {safe_format(relevant.get('handover_improves_tp_flag'))}
• Poor TP: {safe_format(relevant.get('tp_samples_below_600'), '', 0)} samples <600 Mbps, drop ratio={safe_format(relevant.get('tp_drop_ratio'))}
• Signal: RSRP={rsrp_mean}, SINR={sinr_mean} (both degraded, correlated)
• Pattern: Single dominant neighbor available (not C4 symmetric interference)
**Root Cause:** C3 - Wrong cell selection (handovers consistently improve TP)""",

        "C4": f"""**Analysis:**
• Signal decoupling: RSRP={rsrp_mean} (good), SINR={sinr_mean} (poor) → interference
• Multi-site: Non-colocated strong neighbors={safe_format(relevant.get('noncolocated_strong_neighbor_gnodeb_count_mean'))}
• Non-colocated share: {safe_format(relevant.get('strong_neighbor_noncolocated_share'))}
• Symmetry: Top1-top2 gap={safe_format(relevant.get('top1_top2_gap_db_mean'), ' dB')} (small) → multiple equal competitors
• High interference flag: {safe_format(relevant.get('high_interference_power_ratio_flag'))}
• Best neighbor advantage: {safe_format(relevant.get('rsrp_advantage_of_best_neighbor'), ' dB')} (no clear winner)
**Root Cause:** C4 - Overlapping coverage (multiple non-colocated sites, symmetric interference)""",

        "C5": f"""**Analysis:**
• Ping-pong: {safe_format(relevant.get('ping_pong_handover_count'), '', 0)} detected events
• Frequent HO flag: {safe_format(relevant.get('frequent_handover_flag'))}, ping-pong detected: {safe_format(relevant.get('ping_pong_detected'))}
• Total handovers: {safe_format(relevant.get('handover_count'), '', 0)} (≥3 = frequent)
• HO RSRP delta: {safe_format(relevant.get('handover_rsrp_delta_mean'), ' dB')}
• HO TP delta: {safe_format(relevant.get('handover_tp_delta_mean'), ' Mbps')}
• Pattern: Back-and-forth switching (not unidirectional mobility)
**Root Cause:** C5 - Ping-pong handover (hysteresis/parameter issue)""",

        "C6": f"""**Analysis:**
• PCI config: Serving PCI={safe_format(relevant.get('serving_pci'), '', 0)}
• Neighbors: Colocated={safe_format(relevant.get('colocated_neighbor_count'), '', 0)}, Non-colocated={safe_format(relevant.get('noncolocated_neighbor_count'), '', 0)}
• PCI pattern: Check for mod-30 collisions, PCI reuse, confusion between cells
• Abnormal path loss: {safe_format(relevant.get('abnormal_path_loss'))}
• Tilt config: Electronic={safe_format(relevant.get('serving_electronic_tilt_deg'), '°')}
• Not typical interference (C4) or coverage (C1) pattern → PCI-specific issue
**Root Cause:** C6 - PCI collision/confusion (mod-30 collision, PCI reuse causing cell ID ambiguity)""",

        "C7": f"""**Analysis:**
• Speed: max={safe_format(relevant.get('speed_max_kmh'), ' km/h')} (>40 threshold), mean={safe_format(relevant.get('speed_mean_kmh'), ' km/h')}
• Speed flag: {safe_format(relevant.get('speed_above_40_flag'))}, C7 hint: {safe_format(relevant.get('c7_speed_hint'))}
• Mobility impact: Fast+low TP ratio={safe_format(relevant.get('fast_low_tp_ratio'))}
• Speed-TP correlation: {safe_format(relevant.get('speed_tp_correlation'))}
• Performance degrades at high speed (not static coverage/interference)
• Likely cause: Doppler, tracking issues, HO delays at high mobility
**Root Cause:** C7 - High speed impact (>40 km/h mobility degradation)""",

        "C8": f"""**Analysis:**
• Resources: RB mean={safe_format(relevant.get('rb_mean'))}, RB min={safe_format(relevant.get('rb_min'), '', 0)}
• Efficiency: High RB+low TP ratio={safe_format(relevant.get('high_rb_low_tp_ratio_v2'))}
• TP drop with high RB: {safe_format(relevant.get('tp_drop_with_high_rb_ratio'))}
• RB-TP correlation: {safe_format(relevant.get('rb_tp_correlation'))} (weak/negative)
• TP efficiency min: {safe_format(relevant.get('throughput_efficiency_min'))}
• Pattern: High resource allocation but low throughput realization (not coverage/interference)
**Root Cause:** C8 - Resource congestion (high RB allocation, poor scheduling/backhaul)"""
    }

    return reasoning_templates.get(answer, "**Analysis:** Based on the features provided...")

In [10]:
# ============================================================================
# PRINCIPLE-BASED REASONING: Teaching the Model to Think
# ============================================================================

"""
CORE PHILOSOPHY:
Instead of:
  ❌ Pre-computed features → Model matches patterns
  ❌ Templated reasoning → Model fills in blanks
  
Use:
  ✅ Raw data + Domain principles → Model thinks through problem
  ✅ Teach WHAT to look for, not HOW to say it
  ✅ Model develops its own reasoning path

This is how LLMs should work: Learn principles, apply independently.
"""

def create_principle_based_prompt(row: Dict) -> Dict:
    """
    Create training format that teaches domain principles.
    Model learns to extract data, analyze patterns, and reach conclusions.
    
    CRITICAL: System prompt emphasizes general analytical reasoning applied to a specific domain,
    NOT domain-specific identity. This prevents catastrophic forgetting of base reasoning abilities.
    """
    
    original_q = row['original_question']
    answer = row['answer']
    
    # Universal reasoning prompt with PHYSICS-BASED domain context (NOT domain-locked identity!)
    SYSTEM_PROMPT = """You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting by examining drive test and engineering data.

**GENERAL ANALYTICAL APPROACH:**
1. **Data Extraction** - Identify key metrics from tables and text
2. **Pattern Recognition** - Look for correlations, anomalies, and trends in physical measurements
3. **Physics-Based Hypothesis Formation** - Consider mechanisms explaining observed RF behavior
4. **Logical Elimination** - Rule out hypotheses inconsistent with physical signatures
5. **Evidence-Based Conclusion** - Select explanation best supported by RF physics

**DOMAIN CONTEXT - RF PHYSICS & MEASUREMENTS:**

*Signal Strength & Quality:*
• RSRP (Reference Signal Received Power): Measures signal strength
  - Strong: >-90 dBm | Moderate: -90 to -100 dBm | Weak: <-100 dBm
  - Gradient: How quickly signal decays with distance (normal: -8 to -10 dB/100m)
• SINR (Signal-to-Interference-plus-Noise Ratio): Measures signal quality
  - Good: >15 dB | Fair: 5-15 dB | Poor: <5 dB
  - SINR-RSRP correlation: Strong correlation → coverage-driven | Weak correlation → interference-driven

*Spatial Physics & Antenna Geometry:*
• Signal propagation: Naturally weakens with distance (path loss)
• Antenna downtilt: Vertical beam angle affects coverage footprint
  - Excessive tilt (>8°): Steep near-field, weak far-field (coverage holes)
  - Insufficient tilt (<4°): Extended coverage beyond intended area (overshoot >1km)
• Cell radius: Typical macro cell serves 300m-1km; anomalies indicate configuration issues

*Interference Physics:*
• Co-frequency interference: Multiple cells on same frequency cause SINR degradation
• PCI (Physical Cell ID): Collision/confusion when same ID used by non-colocated cells
• Interference signature: Good RSRP + poor SINR → interference dominant, not coverage
• Overlapping coverage: Neighbor RSRP within 5 dB of serving cell → spatial interference

*Mobility & Handover Physics:*
• Handover: Cell transition as UE moves; triggers when neighbor RSRP exceeds serving + margin
• Ping-pong: Repeated handovers (>0.1 Hz) indicate unstable cell boundary or hysteresis issues
• High-speed fading: UE speed >40 km/h causes rapid channel variations, SINR variance
• Handover impact: Throughput dips, signaling delays during transition

*Resource Scheduling:*
• RB (Resource Blocks): Frequency-time allocation units (max ~273 RBs for 100MHz)
• Low RB allocation (<160): Under-scheduling despite adequate RF conditions
• Throughput efficiency: Mbps per RB; low efficiency → link bottleneck beyond RF

**POSSIBLE ROOT CAUSES WITH PHYSICAL SIGNATURES:**
1. **Excessive Downtilt** → Steep RSRP gradient (>-10 dB/100m), weak far-edge, RSRP-SINR correlated
2. **Overshoot (>1km)** → Extended coverage radius, shallow RSRP decay, neighbor weak near-site
3. **Wrong Cell Selection** → Neighbor RSRP stronger, neighbor throughput better, better geometry
4. **Overlapping Coverage** → Multiple strong neighbors (~5 dB apart), PCI collision, SINR-RSRP decoupled
5. **Ping-Pong Handover** → Rapid PCI changes (>0.1 Hz), SINR variance at transitions, throughput drops
6. **PCI Mod-30 Issue** → Same PCI mod-30 across multiple cells, interference spikes at specific locations
7. **High Speed Impact** → GPS speed >40 km/h, high SINR variance, frequent handovers, consecutive PCI changes
8. **Resource Congestion** → Low scheduled RBs (<160), poor throughput despite good RSRP/SINR

**ANALYTICAL PHYSICS PROCESS:**
1. Check signal levels → Coverage-driven (RSRP weak) or interference-driven (SINR weak)?
2. Examine spatial pattern → Distance gradients, cell radius, neighbor relationships
3. Test interference hypothesis → SINR-RSRP decoupling? PCI collisions? Overlapping signals?
4. Assess mobility → Handover frequency, speed, SINR stability
5. Verify resource allocation → RBs sufficient? Efficiency normal?

Output format: Your step-by-step physical analysis followed by \\boxed{{n}} where n=1..8"""

    # Analytical framing - emphasizes reasoning process
    USER_PROMPT = f"""{original_q}

Apply systematic analytical reasoning to this network performance problem:

1. **Extract key data** - What are the relevant measurements?
2. **Identify patterns** - What trends or anomalies do you observe?
3. **Consider explanations** - Which root causes are plausible?
4. **Eliminate systematically** - Which hypotheses don't fit the evidence?
5. **Conclude with evidence** - Which root cause best explains all observations?

Show your reasoning process, then provide your answer as \\boxed{{n}}."""

    # Generate NATURAL reasoning (not template-based)
    assistant_response = generate_natural_reasoning(answer, row)
    
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
            {"role": "assistant", "content": assistant_response}
        ]
    }


def generate_natural_reasoning(answer: str, row: Dict) -> str:
    """
    Generate natural, human-like reasoning that varies in structure.
    NOT templates - teach the model to think flexibly.
    """
    
    features = row.get('features_dict', {})
    
    # Get ground truth for training, but reason naturally about it
    reasoning_variants = {
        "C1": [
            # Variant 1: Start with observation
            f"""Looking at the signal quality, I notice the RSRP values are weak, particularly at the cell edge. The mean RSRP is around {safe_format(features.get('rsrp_mean_dbm'), ' dBm')}, but what's more telling is the minimum reaches {safe_format(features.get('rsrp_min_dbm'), ' dBm')}, and the 10th percentile is quite low at {safe_format(features.get('rsrp_10th_percentile'), ' dBm')}. This pattern of weak far-end coverage is classic.

When I check the cell geometry, I see the serving cell has a tilt of {safe_format(features.get('serving_total_tilt_deg'), '°')}. Given the distances involved (p95 around {safe_format(features.get('dist_p95_m'), 'm')}), this tilt appears excessive - the antenna is pointing too far down, undershooting distant users.

The SINR degradation of {safe_format(features.get('sinr_degradation_db'), ' dB')} when RSRP is relatively stable also points to a coverage geometry issue rather than interference. If this were interference, we'd see different patterns.

Looking at neighbors, there's no significantly better cell available (best neighbor advantage is only {safe_format(features.get('rsrp_advantage_of_best_neighbor'), ' dB')}), which rules out wrong cell selection.

**Conclusion:** The weak far-end coverage pattern combined with excessive antenna tilt indicates C1 - Excessive Downtilt. The cell's coverage area is compromised by geometry issues.

\\boxed{{1}}""",
            
            # Variant 2: Start with elimination
            f"""Let me work through this systematically. First, I'll check if this is an interference problem - if RSRP was good but SINR poor, that would suggest overlapping coverage (C4). However, I see RSRP mean is {safe_format(features.get('rsrp_mean_dbm'), ' dBm')} with SINR at {safe_format(features.get('sinr_mean_db'), ' dB')} - both are degraded together, so probably not interference.

Could this be wrong cell selection (C3)? That would show up as handovers improving throughput. Let me check... the handover behavior doesn't show significant throughput improvements, and the best neighbor advantage is minimal at {safe_format(features.get('rsrp_advantage_of_best_neighbor'), ' dB')}.

What about distance - is this overshoot (C2)? The p95 distance is {safe_format(features.get('dist_p95_m'), 'm')}, which isn't extreme (overshoot typically >1000m).

The key clue is the distribution of RSRP: the 10th percentile at {safe_format(features.get('rsrp_10th_percentile'), ' dBm')} is much worse than the mean, indicating cell edge weakness. The cell tilt is {safe_format(features.get('serving_total_tilt_deg'), '°')}, which for this coverage area appears too aggressive.

This is a geometry problem - the antenna downtilt is causing poor far-end coverage. C1 - Excessive Downtilt.

\\boxed{{1}}"""
        ],
        
        "C2": [
            f"""The first thing that jumps out is the distance. The UE reaches {safe_format(features.get('dist_max_m'), 'm')} from the serving cell at maximum, with a 95th percentile of {safe_format(features.get('dist_p95_m'), 'm')}. In 5G networks, serving cells beyond 1km usually indicates an overshoot situation.

Let me verify there are other cells available. Looking at the handover count ({safe_format(features.get('handover_count'), '', 0)} handovers), yes, multiple cells are visible. The throughput variance across cells is {safe_format(features.get('throughput_variance_across_cells'), ' Mbps')}, suggesting some cells perform better than others. In fact, the best alternative cell would provide {safe_format(features.get('best_cell_throughput_gap'), ' Mbps')} more throughput.

So we have: distant serving cell + better alternatives available = wrong serving area assignment.

This is different from C3 (wrong cell selection) because the issue is fundamentally about distance and coverage boundaries, not about indoor penetration or temporary conditions. The UE is simply too far from its serving cell.

**Diagnosis:** C2 - Overshoot. The UE should be served by a closer cell.

\\boxed{{2}}""",
            
            f"""Distance is the story here. When I calculate the UE's position relative to the serving cell, I get distances up to {safe_format(features.get('dist_max_m'), 'm')}, with p95 at {safe_format(features.get('dist_p95_m'), 'm')}. That's definitely in overshoot territory (typically we flag >1000m).

But let me make sure this isn't just weak coverage everywhere. If it were C1 (excessive downtilt), we'd see weak edge coverage but the UE would still be within normal range. Here, the UE has actually traveled quite far from the cell.

Are there handovers happening? Yes, {safe_format(features.get('handover_count'), '', 0)} times. This confirms multiple cells are in view. The throughput varies significantly across cells ({safe_format(features.get('throughput_variance_across_cells'), ' Mbps')} variance), meaning better options exist.

The pattern is clear: distant serving cell + available alternatives = serving area boundary issue.

C2 - Overshoot.

\\boxed{{2}}"""
        ],
        
        "C3": [
            f"""What immediately catches my attention is the handover behavior. Every time the UE switches cells, throughput changes dramatically. The average change is {safe_format(features.get('avg_throughput_change_after_handover'), ' Mbps')}, with handover TP delta at {safe_format(features.get('handover_tp_delta_mean'), ' Mbps')}. That's a significant and consistent improvement.

Let me look at the throughput distribution. There are {safe_format(features.get('tp_samples_below_600'), '', 0)} samples below 600 Mbps, with a drop ratio of {safe_format(features.get('tp_drop_ratio'))}. Performance is poor with the initial cell, but improves after handover - this is the signature of wrong cell selection.

Is this interference-related (C4)? Let me check signal patterns. RSRP is {safe_format(features.get('rsrp_mean_dbm'), ' dBm')} and SINR is {safe_format(features.get('sinr_mean_db'), ' dB')} - both degraded together (correlated). If it were interference, RSRP would be okay but SINR poor. This pattern suggests penetration loss or obstruction, not multi-site interference.

The cell selection is simply wrong from the start. The UE should have been on the neighbor cell, which consistently provides better service.

C3 - Wrong Cell Selection.

\\boxed{{3}}""",
            
            f"""The throughput pattern here tells an interesting story. Performance is consistently poor ({safe_format(features.get('tp_samples_below_600'), '', 0)} samples below 600 Mbps), but then improves dramatically after handovers. The average throughput change after handover is {safe_format(features.get('avg_throughput_change_after_handover'), ' Mbps')} - that's not random variation, that's a clear indication of wrong initial cell selection.

Why is the initial cell selection wrong? Looking at the signal patterns (RSRP {safe_format(features.get('rsrp_mean_dbm'), ' dBm')}, SINR {safe_format(features.get('sinr_mean_db'), ' dB')}), both are degraded together. This could be due to:
- Indoor penetration loss (signal weakened through walls)
- Physical obstruction
- Signal propagation characteristics favoring the neighbor

The key differentiator from C4 (overlapping coverage) is that we have one dominant better neighbor, not multiple competing cells with symmetric interference.

This is C3 - the UE is served by the wrong cell. Handovers fix the problem by switching to the correct cell.

\\boxed{{3}}"""
        ],
        
        "C4": [
            f"""The signal quality here shows an interesting decoupling pattern. RSRP is {safe_format(features.get('rsrp_mean_dbm'), ' dBm')} - that's actually reasonable signal strength. But SINR is {safe_format(features.get('sinr_mean_db'), ' dB')} - quite poor. This decoupling (good signal strength + poor signal quality) is the hallmark of interference.

Where's the interference coming from? Let me count the strong neighbors. On average, {safe_format(features.get('noncolocated_strong_neighbor_gnodeb_count_mean'))} strong neighbors from different sites (non-co-located) are visible simultaneously. The non-co-located share is {safe_format(features.get('strong_neighbor_noncolocated_share'))} - meaning most strong neighbors are from different physical sites, not the same tower.

This creates a symmetric interference pattern. The gap between the top two neighbors is only {safe_format(features.get('top1_top2_gap_db_mean'), ' dB')} - there's no clear winner. Multiple cells are competing equally for coverage in this area.

The high interference power ratio flag confirms this: {safe_format(features.get('high_interference_power_ratio_flag'))}.

This is classic overlapping coverage from multiple sites. C4.

\\boxed{{4}}""",
            
            f"""Let me diagnose this interference issue. The telltale sign is right in the RSRP/SINR relationship: RSRP at {safe_format(features.get('rsrp_mean_dbm'), ' dBm')} (adequate) but SINR at {safe_format(features.get('sinr_mean_db'), ' dB')} (poor). When you have good signal strength but poor signal quality, you're looking at interference.

The question is: what type of interference? Let me analyze the neighbor situation. There are {safe_format(features.get('noncolocated_strong_neighbor_gnodeb_count_mean'))} non-co-located gNodeBs providing strong signals. The key word here is "non-co-located" - these are different physical sites, not sectors from the same tower.

The neighbor distribution shows small gaps - top1 to top2 gap is only {safe_format(features.get('top1_top2_gap_db_mean'), ' dB')}. This means multiple cells are equally strong, creating symmetric interference. No single cell dominates, so we can't just hand over to a better cell (which would be C3).

The interference pattern, combined with multiple non-co-located strong neighbors, points definitively to C4 - Overlapping Coverage. Multiple sites are providing service to the same area, causing interference.

\\boxed{{4}}"""
        ],
        
        "C5": [
            f"""The handover pattern here is unusual. I'm seeing {safe_format(features.get('handover_count'), '', 0)} total handovers, but more importantly, {safe_format(features.get('ping_pong_handover_count'), '', 0)} of these are ping-pong handovers - meaning the UE switches back and forth between the same cells repeatedly.

Let me verify this is truly ping-pong and not just normal mobility. Looking at the handover deltas: RSRP changes by {safe_format(features.get('handover_rsrp_delta_mean'), ' dB')} and throughput by {safe_format(features.get('handover_tp_delta_mean'), ' Mbps')} on average. These are small changes - if this were wrong cell selection (C3), we'd see large improvements. Instead, we're seeing back-and-forth switching with minimal benefit.

The frequent handover flag is {safe_format(features.get('frequent_handover_flag'))}, and ping-pong is definitely detected: {safe_format(features.get('ping_pong_detected'))}.

This is not high-speed mobility (C7), which would be unidirectional with speed >40 km/h. This is a stationary or slow-moving scenario with poor handover parameter tuning - likely hysteresis is too small or A3 offsets are misconfigured.

C5 - Ping-Pong Handover.

\\boxed{{5}}""",
            
            f"""There's excessive handover activity here. {safe_format(features.get('handover_count'), '', 0)} handovers, with {safe_format(features.get('ping_pong_handover_count'), '', 0)} identified as ping-pong patterns. That's the key diagnostic - not just frequent handovers, but back-and-forth switching.

Why is this happening? When cells have similar signal strength and handover parameters aren't properly set (hysteresis, time-to-trigger), the UE can oscillate between cells. Each handover has minimal RSRP delta ({safe_format(features.get('handover_rsrp_delta_mean'), ' dB')}) and throughput delta ({safe_format(features.get('handover_tp_delta_mean'), ' Mbps')}), confirming the cells are similar and neither provides clear advantage.

This differs from C3 (wrong cell selection) where handovers improve performance, and C7 (high speed) where handovers are due to mobility, not parameter issues.

The ping-pong behavior indicates handover parameter tuning problems. C5.

\\boxed{{5}}"""
        ],
        
        "C6": [
            f"""This case has some unusual characteristics that don't fit the typical coverage or interference patterns. Let me investigate the configuration.

The serving PCI is {safe_format(features.get('serving_pci'), '', 0)}. Looking at the neighbor count: {safe_format(features.get('colocated_neighbor_count'), '', 0)} co-located neighbors and {safe_format(features.get('noncolocated_neighbor_count'), '', 0)} non-co-located. 

The abnormal path loss indicator shows {safe_format(features.get('abnormal_path_loss'))}, and the electronic tilt is {safe_format(features.get('serving_electronic_tilt_deg'), '°')}. But these configuration values don't explain the performance issue in the typical C1 (downtilt) way.

What I'm suspecting is a PCI-related problem. PCI collisions occur when:
1. Same PCI used in overlapping coverage areas (PCI collision)
2. PCIs sharing the same value modulo 30 (PCI confusion)
3. Cell ID ambiguity causing measurement/handover issues

The pattern doesn't match typical coverage (C1/C2), interference (C4), or cell selection (C3) issues. The symptoms suggest systematic configuration problems affecting cell identification and measurement reporting.

C6 - PCI Collision or Configuration Issue.

\\boxed{{6}}""",
            
            f"""Let me think through what's unusual here. The signal patterns don't quite fit interference (C4) - we'd see good RSRP with poor SINR. They don't fit coverage issues (C1/C2) - the distance and tilt don't explain everything. And handovers don't show the clear improvement pattern of C3.

The serving PCI is {safe_format(features.get('serving_pci'), '', 0)}. I need to consider whether there's a PCI reuse issue in this area. PCI planning is critical in 5G/LTE - when PCIs collide or share the same mod-30 value, UEs can't properly distinguish cells. This causes:
- Incorrect neighbor cell measurements
- Handover failures or delays
- Cell reselection problems
- General confusion in cell identification

The neighbor counts ({safe_format(features.get('colocated_neighbor_count'), '', 0)} colocated, {safe_format(features.get('noncolocated_neighbor_count'), '', 0)} non-colocated) and configuration parameters suggest this is a systematic network planning issue rather than a propagation or load problem.

This points to C6 - PCI-related configuration issues.

\\boxed{{6}}"""
        ],
        
        "C7": [
            f"""The speed data here is critical. Maximum speed reaches {safe_format(features.get('speed_max_kmh'), ' km/h')}, with mean at {safe_format(features.get('speed_mean_kmh'), ' km/h')}. The 40 km/h threshold for high-speed issues is clearly exceeded.

Now, is speed actually correlated with performance degradation? The fast+low throughput ratio is {safe_format(features.get('fast_low_tp_ratio'))}, and the speed-TP correlation is {safe_format(features.get('speed_tp_correlation'))}. This confirms that performance specifically degrades at high speeds.

Why does high speed cause problems?
- Doppler effect: Frequency shifts make channel estimation harder
- Rapid cell changes: Handover procedures can't keep up
- Tracking loops: UE and network struggle to maintain synchronization
- Measurement averaging: Fast changes make measurements less reliable

If this were a static coverage issue (C1/C2) or interference (C4), speed wouldn't be a factor - we'd see problems at all speeds. The speed-correlated degradation is diagnostic for C7.

High Speed Impact - C7.

\\boxed{{7}}""",
            
            f"""I'm seeing speed values that are concerning: max {safe_format(features.get('speed_max_kmh'), ' km/h')}, mean {safe_format(features.get('speed_mean_kmh'), ' km/h')}. In 5G, speeds above 40 km/h start causing issues with Doppler compensation and tracking.

Let me verify this is truly speed-related and not coincidental. The speed above 40 flag is {safe_format(features.get('speed_above_40_flag'))}, and the C7 speed hint indicator is {safe_format(features.get('c7_speed_hint'))}. The fast+low TP ratio of {safe_format(features.get('fast_low_tp_ratio'))} shows that low throughput specifically occurs during high-speed periods.

This rules out:
- C1/C2 (coverage): Would affect all speeds equally
- C4 (interference): Location-dependent, not speed-dependent  
- C5 (ping-pong): Configuration issue, not mobility issue

The speed-performance correlation is clear. This is high-speed mobility degradation.

C7.

\\boxed{{7}}"""
        ],
        
        "C8": [
            f"""The resource allocation pattern here is telling. RB (resource block) mean allocation is {safe_format(features.get('rb_mean'))}, with minimum at {safe_format(features.get('rb_min'), '', 0)}. That's high resource allocation.

But what about throughput? That's where it gets interesting. The high RB + low TP ratio is {safe_format(features.get('high_rb_low_tp_ratio_v2'))}, indicating poor efficiency. We also see TP dropping with high RB at a ratio of {safe_format(features.get('tp_drop_with_high_rb_ratio'))}.

The RB-TP correlation is {safe_format(features.get('rb_tp_correlation'))} - weak or even negative. Normally, more resources should mean more throughput. When that relationship breaks down, we're looking at:
- Network congestion: Too many users, scheduler overwhelmed
- Backhaul limitations: Radio has resources but backend can't handle traffic
- Poor scheduling algorithms: Resources allocated inefficiently

This isn't a coverage issue (RSRP would be poor) or interference (SINR would be poor). The radio layer has resources but can't convert them to throughput.

C8 - Resource Congestion.

\\boxed{{8}}""",
            
            f"""Let me analyze the resource utilization. The UE is getting plenty of resource blocks - mean RB is {safe_format(features.get('rb_mean'))}, minimum {safe_format(features.get('rb_min'), '', 0)}. So resource starvation isn't the problem.

The problem is efficiency. Despite high RB allocation, throughput efficiency is poor (minimum efficiency: {safe_format(features.get('throughput_efficiency_min'))}). The RB-TP correlation is {safe_format(features.get('rb_tp_correlation'))} - that should be strongly positive (more resources = more throughput), but it's weak or negative here.

What causes this pattern?
- Network overload: Scheduler can't keep up
- Backhaul bottleneck: Core network limiting
- Contention and overhead: Too many users competing
- Poor PRB mapping: Resources not optimally assigned

The key diagnostic is: high resource allocation + poor throughput realization. The radio interface is fine (it's allocating resources), but something downstream is bottlenecked.

This is C8 - Resource Congestion or scheduling inefficiency.

\\boxed{{8}}"""
        ]
    }
    
    # Randomly select a reasoning variant to teach diverse thinking
    import random
    variants = reasoning_variants.get(answer, [f"Analyzing the data... \\boxed{{{answer[1]}}}"])
    return random.choice(variants)


print("✓ Principle-based reasoning strategy created")
print("  Philosophy: Teach principles, let model think")
print("  No templates, no dependencies, pure reasoning")

✓ Principle-based reasoning strategy created
  Philosophy: Teach principles, let model think
  No templates, no dependencies, pure reasoning


## Generate Training Dataset with Principle-Based Reasoning

Now let's create the training data using the principle-based approach. This will generate natural reasoning examples that teach the model to think independently.

In [11]:
# Process training data - extract features from raw questions
print("Processing training data (extracting features)...")
print("=" * 80)

train_processed = []
failed_count = 0

for idx, row in df_train.iterrows():
    try:
        # Parse the question
        question = sanitize_question_text(row['question'])
        answer = row['answer']
        
        # Extract features
        features_dict = extract_all_features(question)
        
        # Store processed row
        train_processed.append({
            'id': row['ID'],
            'original_question': question,
            'features_dict': features_dict,
            'answer': answer
        })
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(df_train)} samples...")
        
    except Exception as e:
        if idx < 5:  # Show first few errors for debugging
            print(f"⚠️  Error processing row {idx} (ID: {row.get('ID', 'N/A')}): {str(e)}")
        failed_count += 1
        continue

print(f"\n{'='*80}")
print(f"✓ Processed {len(train_processed)} training samples")
print(f"  Failed: {failed_count}")

if len(train_processed) == 0:
    print("⚠️  WARNING: No samples were processed successfully!")
    print("   Check error messages above for debugging")
else:
    # Convert to DataFrame for easier handling
    df_train_processed = pd.DataFrame(train_processed)
    print(f"Train DataFrame shape: {df_train_processed.shape}")
    print(f"Sample row columns: {list(train_processed[0].keys())}")
print(f"{'='*80}")

Processing training data (extracting features)...
  Processed 100/2400 samples...
  Processed 200/2400 samples...
  Processed 300/2400 samples...
  Processed 400/2400 samples...
  Processed 500/2400 samples...
  Processed 600/2400 samples...
  Processed 700/2400 samples...
  Processed 800/2400 samples...
  Processed 900/2400 samples...
  Processed 1000/2400 samples...
  Processed 1100/2400 samples...
  Processed 1200/2400 samples...
  Processed 1300/2400 samples...
  Processed 1400/2400 samples...
  Processed 1500/2400 samples...
  Processed 1600/2400 samples...
  Processed 1700/2400 samples...
  Processed 1800/2400 samples...
  Processed 1900/2400 samples...
  Processed 2000/2400 samples...
  Processed 2100/2400 samples...
  Processed 2200/2400 samples...
  Processed 2300/2400 samples...
  Processed 2400/2400 samples...

✓ Processed 2400 training samples
  Failed: 0
Train DataFrame shape: (2400, 4)
Sample row columns: ['id', 'original_question', 'features_dict', 'answer']


In [12]:
# Process training data and generate principle-based prompts
print("Generating principle-based training examples...")
print("=" * 80)

principle_based_records = []
failed_count = 0

for idx, row in df_train_processed.iterrows():
    try:
        # Create the principle-based chat format
        chat_format = create_principle_based_prompt(row)
        
        # Add answer as separate field for easy pandas loading
        chat_format['answer'] = row['answer']
        chat_format['id'] = row['id']
        
        principle_based_records.append(chat_format)
        
        # Show first 2 examples for verification
        if idx < 2:
            print(f"\n{'='*80}")
            print(f"EXAMPLE {idx + 1} (Answer: {row['answer']})")
            print(f"{'='*80}")
            print(f"\nSYSTEM PROMPT (first 300 chars):")
            print(chat_format['messages'][0]['content'][:300] + "...")
            print(f"\nUSER PROMPT (first 250 chars):")
            print(chat_format['messages'][1]['content'][:250] + "...")
            print(f"\nASSISTANT REASONING (first 500 chars):")
            print(chat_format['messages'][2]['content'][:500] + "...")
            print()
            
    except Exception as e:
        print(f"⚠️  Error processing row {idx}: {str(e)}")
        failed_count += 1
        continue

print(f"\n{'='*80}")
print(f"✓ Generated {len(principle_based_records)} training examples")
print(f"  Failed: {failed_count}")
print(f"  Strategy: Pure principle-based reasoning")
print(f"  Model learns: Extract → Analyze → Reason → Conclude")
print(f"  No templates, no dependencies, full autonomy")
print(f"{'='*80}")

Generating principle-based training examples...

EXAMPLE 1 (Answer: C2)

SYSTEM PROMPT (first 300 chars):
You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting by examining drive test and engineering data.

**GENERAL ANALYTICAL APPROACH:**
1. **Data Extraction** - Identify key metrics from table...

USER PROMPT (first 250 chars):
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and ...

ASSISTANT REASONING (first 500 chars):
The first thing that jumps out is the distance. The UE reaches N/A from the serving cell at maximum, with a 95th percentile of N/A. In 5G networks, serving cells beyond 1km usually indicates an overshoot situation.

Let me verify there are other cells available. L

In [13]:
# Export training data to JSONL format
output_file = 'qwen_rca_train_principle_based.jsonl'

with open(output_file, 'w') as f:
    for record in principle_based_records:
        f.write(json.dumps(record) + '\n')

print(f"✓ Exported training data to: {output_file}")
print(f"  Total examples: {len(principle_based_records)}")
print(f"  Format: JSONL with messages array (Qwen chat format)")
print(f"  Structure: system + user + assistant")
print(f"  Ready for: Hugging Face fine-tuning with Qwen model")
print(f"\n{'='*80}")

✓ Exported training data to: qwen_rca_train_principle_based.jsonl
  Total examples: 2400
  Format: JSONL with messages array (Qwen chat format)
  Structure: system + user + assistant
  Ready for: Hugging Face fine-tuning with Qwen model



In [14]:
import pandas as pd

# Load the JSONL file
df = pd.read_json('qwen_rca_train_principle_based.jsonl', lines=True)

# You'll now see columns: 'messages', 'answer', 'id'
print(df.columns)  # ['messages', 'answer', 'id']
df.head() # Easy to view answers!

Index(['messages', 'answer', 'id'], dtype='object')


,messages,answer,id
0,"[{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting ...",C2,ID_1P7PJMPV0R
1,"[{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting ...",C1,ID_8B1D1TUTFA
2,"[{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting ...",C2,ID_IGGXMA9GZH
3,"[{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting ...",C2,ID_D6C9N2X295
4,"[{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting ...",C5,ID_8JC15PNP3Q


In [32]:
df.shape

(2400, 3)

## Generate Test Dataset

For test data, we only include system and user prompts (no assistant response). The fine-tuned model will generate the assistant response during inference.

In [37]:
# Load test data
df_test = pd.read_csv(TEST_FILE)
print(f"✓ Loaded {len(df_test)} test samples")
print(f"Columns: {df_test.columns.tolist()}")

✓ Loaded 864 test samples
Columns: ['ID', 'question']


In [38]:
df_test.head()

,ID,question
0,ID_XLWWVM40IW,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
1,ID_3WB8KX32W3,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
2,ID_R2BEOGLAFW,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
3,ID_6S8OMJD79C,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
4,ID_HO8IC7L32U,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...


In [39]:
df_test_2= pd.read_csv("data/phase_2_test.csv")
df_test_2.head()

,ID,question
0,ID_7W4Q1615FZ,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
1,ID_ZDOP7324ZF,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
2,ID_7NWKCWAHP0,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
3,ID_477BBRFS8A,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...
4,ID_ZPS9DOT2XB,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...


In [47]:
df_test= pd.concat([df_test,df_test_2],ignore_index=True)

In [50]:
df_test.shape

(1727, 2)

In [51]:
# Check test data structure
print("Test data sample:")
print(df_test.iloc[0]['question'][:200])

Test data sample:
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 pote


In [52]:
# Process test data (just need to extract features, no answers)
print("Processing test data...")
test_processed = []
failed_test = 0

for idx, row in df_test.iterrows():
    try:
        # Parse the question
        question = sanitize_question_text(row['question'])
        
        # Extract features
        features_dict = extract_all_features(question)
        
        # Store processed row
        test_processed.append({
            'id': row['ID'],  # Use ID column
            'original_question': question,
            'features_dict': features_dict,
            'answer': 'C1'  # Dummy answer for formatting (not used in test)
        })
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(df_test)} test samples...")
        
    except Exception as e:
        if idx < 5:  # Show first few errors
            print(f"⚠️  Error processing test row {idx}: {str(e)}")
        failed_test += 1
        continue

print(f"✓ Processed {len(test_processed)} test samples (Failed: {failed_test})")

# Convert to DataFrame for easier handling
if len(test_processed) > 0:
    df_test_processed = pd.DataFrame(test_processed)
    print(f"Test DataFrame shape: {df_test_processed.shape}")
else:
    print("⚠️  WARNING: No test samples were processed!")

Processing test data...
  Processed 100/1727 test samples...
  Processed 200/1727 test samples...
  Processed 300/1727 test samples...
  Processed 400/1727 test samples...
  Processed 500/1727 test samples...
  Processed 600/1727 test samples...
  Processed 700/1727 test samples...
  Processed 800/1727 test samples...
  Processed 900/1727 test samples...
  Processed 1000/1727 test samples...
  Processed 1100/1727 test samples...
  Processed 1200/1727 test samples...
  Processed 1300/1727 test samples...
  Processed 1400/1727 test samples...
  Processed 1500/1727 test samples...
  Processed 1600/1727 test samples...
  Processed 1700/1727 test samples...
✓ Processed 1727 test samples (Failed: 0)
Test DataFrame shape: (1727, 4)


In [53]:
# Check test processing result
print(f"Test processing result:")
print(f"  test_processed length: {len(test_processed)}")
if len(test_processed) > 0:
    df_test_processed = pd.DataFrame(test_processed)
    print(f"  df_test_processed shape: {df_test_processed.shape}")
    print(f"  Sample keys: {list(test_processed[0].keys())}")
else:
    print("  WARNING: No test samples were processed!")

Test processing result:
  test_processed length: 1727
  df_test_processed shape: (1727, 4)
  Sample keys: ['id', 'original_question', 'features_dict', 'answer']


In [54]:
# Generate test dataset (system + user only, no assistant)
print("Generating principle-based test examples...")
print("=" * 80)

principle_based_test_records = []
failed_test_count = 0

for idx, row in df_test_processed.iterrows():
    try:
        # Use same prompt generation but exclude assistant response
        chat_format = create_principle_based_prompt(row)
        
        # Create test format with only system + user
        test_format = {
            "id": row['id'],  # Keep ID for submission
            "messages": [
                chat_format['messages'][0],  # system
                chat_format['messages'][1]   # user
            ]
        }
        principle_based_test_records.append(test_format)
        
        # Show first example
        if idx == 0:
            print(f"\nTEST EXAMPLE (ID: {row['id']})")
            print(f"{'='*80}")
            print(f"SYSTEM PROMPT (first 300 chars):")
            print(test_format['messages'][0]['content'][:300] + "...")
            print(f"\nUSER PROMPT (first 250 chars):")
            print(test_format['messages'][1]['content'][:250] + "...")
            print(f"\n(No assistant response - model will generate this)")
            print()
            
    except Exception as e:
        print(f"⚠️  Error processing test row {idx}: {str(e)}")
        failed_test_count += 1
        continue

print(f"\n{'='*80}")
print(f"✓ Generated {len(principle_based_test_records)} test examples")
print(f"  Failed: {failed_test_count}")
print(f"  Format: System + User only (model generates assistant response)")
print(f"{'='*80}")

Generating principle-based test examples...

TEST EXAMPLE (ID: ID_XLWWVM40IW)
SYSTEM PROMPT (first 300 chars):
You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting by examining drive test and engineering data.

**GENERAL ANALYTICAL APPROACH:**
1. **Data Extraction** - Identify key metrics from table...

USER PROMPT (first 250 chars):
Analyze the 5G wireless network drive-test user plane data and engineering parameters.
Identify the reason for the throughput dropping below 600Mbps in certain road sections.
From the following 8 potential root causes, select the most likely one and ...

(No assistant response - model will generate this)


✓ Generated 1727 test examples
  Failed: 0
  Format: System + User only (model generates assistant response)


In [55]:
df_test_processed["text"] = principle_based_test_records
df_test_processed.head()

,id,original_question,features_dict,answer,text
0,ID_XLWWVM40IW,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,"{'rsrp_min_dbm': -89.14, 'rsrp_mean_dbm': -85.813, 'rsrp_10th_percentile': -87.76, 'sinr_degradation_db': -0.15100000000000158, 'sinr_std_when_rsrp_stable': 0.9890640571492585, 'dist_max_m': None,...",C1,"{'id': 'ID_XLWWVM40IW', 'messages': [{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis ..."
1,ID_3WB8KX32W3,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,"{'rsrp_min_dbm': -88.32, 'rsrp_mean_dbm': -84.95, 'rsrp_10th_percentile': -87.48, 'sinr_degradation_db': 3.274999999999997, 'sinr_std_when_rsrp_stable': 0.821310230508862, 'dist_max_m': None, 'dis...",C1,"{'id': 'ID_3WB8KX32W3', 'messages': [{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis ..."
2,ID_R2BEOGLAFW,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,"{'rsrp_min_dbm': -88.95, 'rsrp_mean_dbm': -81.74, 'rsrp_10th_percentile': -88.61, 'sinr_degradation_db': -1.6029999999999944, 'sinr_std_when_rsrp_stable': 1.3930758242046772, 'dist_max_m': None, '...",C1,"{'id': 'ID_R2BEOGLAFW', 'messages': [{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis ..."
3,ID_6S8OMJD79C,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,"{'rsrp_min_dbm': -90.85, 'rsrp_mean_dbm': -84.293, 'rsrp_10th_percentile': -89.4, 'sinr_degradation_db': 13.992999999999993, 'sinr_std_when_rsrp_stable': 0.7358500134616461, 'dist_max_m': None, 'd...",C1,"{'id': 'ID_6S8OMJD79C', 'messages': [{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis ..."
4,ID_HO8IC7L32U,Analyze the 5G wireless network drive-test user plane data and engineering parameters.\nIdentify the reason for the throughput dropping below 600Mbps in certain road sections.\nFrom the following ...,"{'rsrp_min_dbm': -91.36, 'rsrp_mean_dbm': -86.807, 'rsrp_10th_percentile': -90.55, 'sinr_degradation_db': -9.796000000000003, 'sinr_std_when_rsrp_stable': 0.3902041402043241, 'dist_max_m': None, '...",C1,"{'id': 'ID_HO8IC7L32U', 'messages': [{'role': 'system', 'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis ..."


In [56]:
df_test_processed['text'].tail().to_list()[0]["messages"]

[{'role': 'system',
  'content': 'You are a helpful AI assistant with strong analytical reasoning capabilities. For this task, you will apply systematic analysis to wireless network troubleshooting by examining drive test and engineering data.\n\n**GENERAL ANALYTICAL APPROACH:**\n1. **Data Extraction** - Identify key metrics from tables and text\n2. **Pattern Recognition** - Look for correlations, anomalies, and trends in physical measurements\n3. **Physics-Based Hypothesis Formation** - Consider mechanisms explaining observed RF behavior\n4. **Logical Elimination** - Rule out hypotheses inconsistent with physical signatures\n5. **Evidence-Based Conclusion** - Select explanation best supported by RF physics\n\n**DOMAIN CONTEXT - RF PHYSICS & MEASUREMENTS:**\n\n*Signal Strength & Quality:*\n• RSRP (Reference Signal Received Power): Measures signal strength\n  - Strong: >-90 dBm | Moderate: -90 to -100 dBm | Weak: <-100 dBm\n  - Gradient: How quickly signal decays with distance (normal: 

In [ ]:
# Export test data to JSONL format
test_output_file = 'qwen_rca_test_principle_based.jsonl'

with open(test_output_file, 'w') as f:
    for record in principle_based_test_records:
        f.write(json.dumps(record) + '\n')

print(f"✓ Exported test data to: {test_output_file}")
print(f"  Total examples: {len(principle_based_test_records)}")
print(f"  Format: JSONL with messages array (Qwen chat format)")
print(f"  Structure: system + user only")
print(f"  Ready for: Inference with fine-tuned Qwen model")
print(f"\n{'='*80}")